# TabDPT Regressor — DIMER-ready smoke tutorial

This notebook verifies the pinned TabDPT v1.2 integration on a small regression task. Public sample metrics are sanity checks only because TabDPT was pretrained on real-world tables and benchmark overlap cannot be ruled out.


In [ ]:
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q '/content/tabdpt-regressor-pipeline[model]'


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from tabdpt_regressor_pipeline import TabDPTRegressionPipeline

frame = load_diabetes(as_frame=True).frame
train, test = train_test_split(frame, test_size=0.2, random_state=42)
print(train.shape, test.shape)


In [ ]:
# Explicitly disable FlashAttention for portability to Colab/Kaggle Tesla T4 (sm_75) GPUs.
pipe = TabDPTRegressionPipeline(compile_model=False, use_flash=False)
pipe.fit(train, target_column='target')
print('ready')


In [ ]:
metrics = pipe.evaluate(test, n_ensembles=2, context_size=512, batch_size=512, seed=42)
metrics


## Production note
For DIMER deployment, mount/cache the verified `tabdpt1_2.safetensors` artifact instead of relying on an internet download. The pipeline auto-enables FlashAttention only on compatible CUDA devices (compute capability 8.0+); callers may still override `use_flash` explicitly. Preserve application-specific train/validation/test splits for leakage-sensitive data.
